# Celebal Technologies Internship - Week 5

## Apache Spark (PySpark)

**Name:** Abhishek Raj

**Internship:** Celebal Technologies Data Engineering Internship

**Week:** 5

---

### Project Objective

This notebook demonstrates various Apache Spark (PySpark) operations including:

- Data Cleaning
- Removing Duplicates
- Handling Missing Values
- Filtering
- Aggregations
- GroupBy
- Data Type Conversion
- Spark Transformations
- Spark Actions

using a custom Retail Sales dataset.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    avg,
    sum,
    count,
    min,
    max,
    when
)
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("Celebal Week 5") \
    .getOrCreate()

spark

## Creating a Custom Retail Sales Dataset

In this section, we generate a custom retail sales dataset containing realistic transaction records.

The dataset includes:
- Duplicate records
- Null values
- Empty usernames
- Multiple regions
- Multiple stores
- Different product categories

This dataset will be used throughout the notebook to answer all Week 5 assignment questions.

In [3]:
import pandas as pd
import random
from datetime import datetime, timedelta
import os

In [4]:
random.seed(42)

In [5]:
regions = ["West", "East", "North", "South"]

cities = {
    "West": ["Los Angeles", "San Francisco", "Seattle"],
    "East": ["New York", "Boston", "Philadelphia"],
    "North": ["Chicago", "Detroit", "Minneapolis"],
    "South": ["Dallas", "Houston", "Miami"]
}

states = {
    "Los Angeles": "CA",
    "San Francisco": "CA",
    "Seattle": "WA",
    "New York": "NY",
    "Boston": "MA",
    "Philadelphia": "PA",
    "Chicago": "IL",
    "Detroit": "MI",
    "Minneapolis": "MN",
    "Dallas": "TX",
    "Houston": "TX",
    "Miami": "FL"
}

categories = [
    "Electronics",
    "Furniture",
    "Books",
    "Clothing",
    "Sports"
]

products = {
    "Electronics": ["Laptop", "Phone", "Monitor", "Keyboard", "Mouse"],
    "Furniture": ["Chair", "Table", "Desk", "Sofa"],
    "Books": ["Python Book", "SQL Guide", "Spark Handbook"],
    "Clothing": ["Jacket", "T-Shirt", "Jeans"],
    "Sports": ["Football", "Cricket Bat", "Tennis Racket"]
}

status_list = ["Delivered", "Pending", "Cancelled"]

subscriptions = ["Premium", "Basic"]

In [7]:
start_date = datetime(2024, 1, 1)

In [8]:
region = random.choice(regions)
city = random.choice(cities[region])
state = states[city]

category = random.choice(categories)
product = random.choice(products[category])

quantity = random.randint(1, 5)

price = float(f"{random.uniform(20,1500):.2f}")
sale_amount = float(f"{price*quantity:.2f}")

transaction_date = (
    start_date +
    timedelta(days=random.randint(0,180))
).date()

timestamp = (
    start_date +
    timedelta(
        days=random.randint(0,180),
        hours=random.randint(0,23),
        minutes=random.randint(0,59)
    )
)

row = {
    "user_id": "U001",
    "transaction_date": transaction_date,
    "transaction_id": "T0001",
    "store_id": "S001",
    "region": region,
    "city": city,
    "state": state,
    "product_category": category,
    "product_name": product,
    "sale_amount": sale_amount,
    "price": price,
    "quantity": quantity,
    "status": random.choice(status_list),
    "age": random.randint(18,45),
    "subscription": random.choice(subscriptions),
    "email": "user1@gmail.com",
    "username": "user1",
    "raw_timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S")
}

print(row)

{'user_id': 'U001', 'transaction_date': datetime.date(2024, 1, 8), 'transaction_id': 'T0001', 'store_id': 'S001', 'region': 'West', 'city': 'Seattle', 'state': 'WA', 'product_category': 'Sports', 'product_name': 'Football', 'sale_amount': 3222.2, 'price': 644.44, 'quantity': 5, 'status': 'Cancelled', 'age': 37, 'subscription': 'Premium', 'email': 'user1@gmail.com', 'username': 'user1', 'raw_timestamp': '2024-01-24 06:14:00'}


In [9]:
rows = []

for i in range(1, 501):

    region = random.choice(regions)
    city = random.choice(cities[region])
    state = states[city]

    category = random.choice(categories)
    product = random.choice(products[category])

    quantity = random.randint(1, 5)

    price = float(f"{random.uniform(20,1500):.2f}")
    sale_amount = float(f"{price * quantity:.2f}")

    transaction_date = (
        start_date +
        timedelta(days=random.randint(0,180))
    ).date()

    timestamp = (
        start_date +
        timedelta(
            days=random.randint(0,180),
            hours=random.randint(0,23),
            minutes=random.randint(0,59)
        )
    )

    row = {
        "user_id": f"U{random.randint(1,100):03}",
        "transaction_date": transaction_date,
        "transaction_id": f"T{i:05}",
        "store_id": f"S{random.randint(1,5):03}",
        "region": region,
        "city": city,
        "state": state,
        "product_category": category,
        "product_name": product,
        "sale_amount": sale_amount,
        "price": price,
        "quantity": quantity,
        "status": random.choice(status_list),
        "age": random.randint(18,45),
        "subscription": random.choice(subscriptions),
        "email": f"user{i}@gmail.com",
        "username": f"user{i}",
        "raw_timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S")
    }

    rows.append(row)

print("Rows Generated:", len(rows))

Rows Generated: 500


In [10]:
# Duplicate records
rows.extend(rows[:20])

# Null Prices
for i in random.sample(range(len(rows)), 20):
    rows[i]["price"] = None

# Null Status
for i in random.sample(range(len(rows)), 15):
    rows[i]["status"] = None

# Null Email
for i in random.sample(range(len(rows)), 15):
    rows[i]["email"] = None

# Empty Username
for i in random.sample(range(len(rows)), 10):
    rows[i]["username"] = ""

print("Final Rows:", len(rows))

Final Rows: 520


In [11]:
df_pd = pd.DataFrame(rows)

df_pd.head()

,user_id,transaction_date,transaction_id,store_id,region,city,state,product_category,product_name,sale_amount,price,quantity,status,age,subscription,email,username,raw_timestamp
0,U055,2024-03-12,T00001,S003,East,Philadelphia,PA,Sports,Cricket Bat,1369.66,NaN,2,Pending,22,Premium,user1@gmail.com,user1,2024-01-02 05:44:00
1,U006,2024-03-29,T00002,S004,North,Chicago,IL,Electronics,Keyboard,551.29,551.29,1,Cancelled,21,Basic,user2@gmail.com,user2,2024-06-03 08:51:00
2,U009,2024-04-02,T00003,S001,West,Seattle,WA,Books,Spark Handbook,6652.35,1330.47,5,Cancelled,25,Basic,user3@gmail.com,user3,2024-05-27 06:45:00
3,U027,2024-04-03,T00004,S003,West,Los Angeles,CA,Electronics,Keyboard,2073.15,691.05,3,Cancelled,39,Premium,user4@gmail.com,user4,2024-02-11 11:22:00
4,U088,2024-06-12,T00005,S003,East,Philadelphia,PA,Furniture,Table,2326.36,581.59,4,Delivered,25,Premium,user5@gmail.com,user5,2024-06-25 17:14:00


In [13]:
os.makedirs("../data", exist_ok=True)

df_pd.to_csv("../data/retail_sales.csv", index=False)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!


## Load the Retail Sales Dataset

The custom retail sales dataset generated in the previous section is loaded into a PySpark DataFrame for further analysis and processing.

In [14]:
df = spark.read.csv(
    "../data/retail_sales.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------+----------------+--------------+--------+------+------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+---------------+--------+-------------------+
|user_id|transaction_date|transaction_id|store_id|region|        city|state|product_category|  product_name|sale_amount|  price|quantity|   status|age|subscription|          email|username|      raw_timestamp|
+-------+----------------+--------------+--------+------+------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+---------------+--------+-------------------+
|   U055|      2024-03-12|        T00001|    S003|  East|Philadelphia|   PA|          Sports|   Cricket Bat|    1369.66|   NULL|       2|  Pending| 22|     Premium|user1@gmail.com|   user1|2024-01-02 05:44:00|
|   U006|      2024-03-29|        T00002|    S004| North|     Chicago|   IL|     Electronics|      Keyboard|     551.29| 551.29|       1|Cancelled| 21|       Ba

## Dataset Overview

The following code displays the total number of records and columns present in the dataset.

In [15]:
print(f"Total Rows    : {df.count()}")
print(f"Total Columns : {len(df.columns)}")

Total Rows    : 520
Total Columns : 18


## Dataset Schema

The schema provides the data type of each column in the dataset.

In [16]:
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



## Column Names

The following command lists all the columns available in the dataset.

In [17]:
df.columns

['user_id',
 'transaction_date',
 'transaction_id',
 'store_id',
 'region',
 'city',
 'state',
 'product_category',
 'product_name',
 'sale_amount',
 'price',
 'quantity',
 'status',
 'age',
 'subscription',
 'email',
 'username',
 'raw_timestamp']

## Missing Value Analysis

The following code checks the number of null values present in each column.

In [18]:
from pyspark.sql.functions import col, when, count

missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show()

+-------+----------------+--------------+--------+------+----+-----+----------------+------------+-----------+-----+--------+------+---+------------+-----+--------+-------------+
|user_id|transaction_date|transaction_id|store_id|region|city|state|product_category|product_name|sale_amount|price|quantity|status|age|subscription|email|username|raw_timestamp|
+-------+----------------+--------------+--------+------+----+-----+----------------+------------+-----------+-----+--------+------+---+------------+-----+--------+-------------+
|      0|               0|             0|       0|     0|   0|    0|               0|           0|          0|   21|       0|    18|  0|           0|   17|      11|            0|
+-------+----------------+--------------+--------+------+----+-----+----------------+------------+-----------+-----+--------+------+---+------------+-----+--------+-------------+



## Duplicate Record Analysis

This step compares the total number of records with the number of unique records.

In [19]:
total_records = df.count()
unique_records = df.dropDuplicates().count()

print("Total Records    :", total_records)
print("Unique Records   :", unique_records)
print("Duplicate Records:", total_records - unique_records)

Total Records    : 520
Unique Records   : 500
Duplicate Records: 20


## Summary Statistics

The following summary provides descriptive statistics for numerical columns.

In [20]:
df.describe().show()

+-------+-------+--------------+--------+------+-------+-----+----------------+-------------+------------------+-----------------+------------------+---------+------------------+------------+-----------------+--------+
|summary|user_id|transaction_id|store_id|region|   city|state|product_category| product_name|       sale_amount|            price|          quantity|   status|               age|subscription|            email|username|
+-------+-------+--------------+--------+------+-------+-----+----------------+-------------+------------------+-----------------+------------------+---------+------------------+------------+-----------------+--------+
|  count|    520|           520|     520|   520|    520|  520|             520|          520|               520|              499|               520|      502|               520|         520|              503|     509|
|   mean|   NULL|          NULL|    NULL|  NULL|   NULL| NULL|            NULL|         NULL| 2278.390826923076|761.77168336

## Create Temporary SQL View

A temporary SQL view is created to enable SQL queries on the DataFrame.

In [21]:
df.createOrReplaceTempView("retail_sales")

ASSIGNMENT


# Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

## Answer

Traditional MapReduce processes data in multiple stages and writes intermediate results to disk after every Map and Reduce operation. This disk-based processing introduces high latency and makes it inefficient for iterative and real-time workloads.

Key limitations of MapReduce include:

- High disk I/O due to writing intermediate results to disk.
- Slow execution for iterative machine learning algorithms.
- Limited support for interactive data analytics.
- Complex programming model requiring separate Map and Reduce functions.
- Not suitable for real-time or streaming applications.

Apache Spark overcomes these limitations by:

- Performing in-memory computation, which significantly improves processing speed.
- Using a Directed Acyclic Graph (DAG) execution engine for optimized task scheduling.
- Providing high-level APIs in Python, Scala, Java, and R.
- Supporting batch processing, SQL, machine learning, graph processing, and stream processing within a single framework.

# Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

## Answer

Apache Spark stores intermediate data in memory (RAM) instead of repeatedly reading from and writing to disk. During iterative algorithms such as machine learning and graph processing, the same dataset is accessed multiple times.

Since Spark keeps frequently used data in memory, it avoids expensive disk I/O operations, resulting in significantly faster execution. Spark also allows datasets to be cached or persisted, enabling efficient reuse across multiple computations.

Compared to traditional disk-based systems like MapReduce, Spark provides much lower latency and better performance for iterative workloads.

# Q3. Remove duplicate rows based on **user_id** and **transaction_date**.

In [22]:
duplicate_removed = df.dropDuplicates(
    ["user_id", "transaction_date"]
)

duplicate_removed.show(10)

+-------+----------------+--------------+--------+------+------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+-----------------+--------+-------------------+
|user_id|transaction_date|transaction_id|store_id|region|        city|state|product_category|  product_name|sale_amount|  price|quantity|   status|age|subscription|            email|username|      raw_timestamp|
+-------+----------------+--------------+--------+------+------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+-----------------+--------+-------------------+
|   U001|      2024-01-20|        T00185|    S004|  East|Philadelphia|   PA|     Electronics|         Mouse|    1261.82|1261.82|       1|Delivered| 44|     Premium|user185@gmail.com| user185|2024-06-20 01:48:00|
|   U001|      2024-02-15|        T00291|    S004|  East|      Boston|   MA|       Furniture|         Chair|    2049.16| 512.29|       4|  Pending| 24| 

# Q4. Filter the dataset for the **West** region and calculate the average **sale_amount** for each **product_category**.

In [23]:
from pyspark.sql.functions import avg

west_sales = (
    df.filter(col("region") == "West")
      .groupBy("product_category")
      .agg(
          avg("sale_amount").alias("Average_Sale")
      )
)

west_sales.show()

+----------------+------------------+
|product_category|      Average_Sale|
+----------------+------------------+
|          Sports|         2020.9392|
|     Electronics|1841.6993939393938|
|        Clothing| 2555.372272727272|
|           Books|2491.0693548387094|
|       Furniture|2019.1579166666663|
+----------------+------------------+



# Q5. What is the difference between `.na.drop()` and `.na.fill()`?

## Answer

- **`.na.drop()`** removes rows containing null values.
- **`.na.fill()`** replaces null values with a specified value without removing any rows.

Using `.na.fill()` helps preserve the dataset while handling missing values.

In [24]:
filled_df = df.na.fill({
    "status": "Unknown"
})

filled_df.select("status").show(10)

+---------+
|   status|
+---------+
|  Pending|
|Cancelled|
|Cancelled|
|Cancelled|
|Delivered|
|  Unknown|
|  Pending|
|Delivered|
|Cancelled|
|Delivered|
+---------+
only showing top 10 rows


# Q6. Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [25]:
from pyspark.sql.functions import count

city_count = (
    df.groupBy("city")
      .agg(count("*").alias("Total"))
      .filter(col("Total") > 100)
)

city_count.show()

+----+-----+
|city|Total|
+----+-----+
+----+-----+



# Q7. How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping columns or renaming them?

## Answer

When performing data cleaning operations such as dropping columns, renaming columns, or filtering rows, Spark creates a new DataFrame containing the updated data while leaving the original DataFrame unchanged.

This design ensures:

- Data consistency
- Fault tolerance
- Easy recovery during distributed processing
- Support for parallel execution

In [26]:
clean_df = df.drop("state")

clean_df.show(5)

+-------+----------------+--------------+--------+------+------------+----------------+--------------+-----------+-------+--------+---------+---+------------+---------------+--------+-------------------+
|user_id|transaction_date|transaction_id|store_id|region|        city|product_category|  product_name|sale_amount|  price|quantity|   status|age|subscription|          email|username|      raw_timestamp|
+-------+----------------+--------------+--------+------+------------+----------------+--------------+-----------+-------+--------+---------+---+------------+---------------+--------+-------------------+
|   U055|      2024-03-12|        T00001|    S003|  East|Philadelphia|          Sports|   Cricket Bat|    1369.66|   NULL|       2|  Pending| 22|     Premium|user1@gmail.com|   user1|2024-01-02 05:44:00|
|   U006|      2024-03-29|        T00002|    S004| North|     Chicago|     Electronics|      Keyboard|     551.29| 551.29|       1|Cancelled| 21|       Basic|user2@gmail.com|   user2|2

# Q8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [27]:
premium_users = df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

premium_users.show()

+-------+----------------+--------------+--------+------+-------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+----------------+--------+-------------------+
|user_id|transaction_date|transaction_id|store_id|region|         city|state|product_category|  product_name|sale_amount|  price|quantity|   status|age|subscription|           email|username|      raw_timestamp|
+-------+----------------+--------------+--------+------+-------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+----------------+--------+-------------------+
|   U055|      2024-03-12|        T00001|    S003|  East| Philadelphia|   PA|          Sports|   Cricket Bat|    1369.66|   NULL|       2|  Pending| 22|     Premium| user1@gmail.com|   user1|2024-01-02 05:44:00|
|   U088|      2024-06-12|        T00005|    S003|  East| Philadelphia|   PA|       Furniture|         Table|    2326.36| 581.59|       4|Delivered| 25|

# Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

## Answer

Handling null values before aggregation helps:

- Improve calculation accuracy.
- Prevent unexpected null results.
- Avoid misleading averages and totals.
- Maintain data quality during analysis.

Replacing or removing null values ensures reliable statistical calculations.

# Q10. Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [28]:
from pyspark.sql.types import TimestampType

timestamp_df = (
    df.withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
)

timestamp_df.show(5)

+-------+----------------+--------------+--------+------+------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+---------------+--------+-------------------+
|user_id|transaction_date|transaction_id|store_id|region|        city|state|product_category|  product_name|sale_amount|  price|quantity|   status|age|subscription|          email|username|         event_time|
+-------+----------------+--------------+--------+------+------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+---------------+--------+-------------------+
|   U055|      2024-03-12|        T00001|    S003|  East|Philadelphia|   PA|          Sports|   Cricket Bat|    1369.66|   NULL|       2|  Pending| 22|     Premium|user1@gmail.com|   user1|2024-01-02 05:44:00|
|   U006|      2024-03-29|        T00002|    S004| North|     Chicago|   IL|     Electronics|      Keyboard|     551.29| 551.29|       1|Cancelled| 21|       Ba

# Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

## Answer

Shuffle is the process of redistributing data across different partitions during operations such as:

- groupBy()
- join()
- distinct()
- reduceByKey()

Since data moves between executors, network communication is required.

It is called a **wide transformation** because records from one partition may be transferred to multiple other partitions before computation continues.

# Q12. Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [29]:
clean_df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

clean_df.show()

+-------+----------------+--------------+--------+------+-------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+----------------+--------+-------------------+
|user_id|transaction_date|transaction_id|store_id|region|         city|state|product_category|  product_name|sale_amount|  price|quantity|   status|age|subscription|           email|username|      raw_timestamp|
+-------+----------------+--------------+--------+------+-------------+-----+----------------+--------------+-----------+-------+--------+---------+---+------------+----------------+--------+-------------------+
|   U055|      2024-03-12|        T00001|    S003|  East| Philadelphia|   PA|          Sports|   Cricket Bat|    1369.66|   NULL|       2|  Pending| 22|     Premium| user1@gmail.com|   user1|2024-01-02 05:44:00|
|   U006|      2024-03-29|        T00002|    S004| North|      Chicago|   IL|     Electronics|      Keyboard|     551.29| 551.29|       1|Cancelled| 21|

# Q13. How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [30]:
price_summary = df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
)

price_summary.show()

+-------------+-------------+-----------------+
|Minimum Price|Maximum Price|    Average Price|
+-------------+-------------+-----------------+
|        24.05|      1499.37|761.7716833667345|
+-------------+-------------+-----------------+



# Q14. In the context of cleaning a dataset, what is the risk of using inferSchema=True when your source data contains messy or inconsistent date formats?

## Answer

Using `inferSchema=True` with inconsistent date formats may lead to:

- Incorrect data type inference.
- Dates being interpreted as strings.
- Invalid values becoming null.
- Inaccurate filtering and aggregations.
- Additional preprocessing before analysis.

For production datasets, defining an explicit schema is often recommended.

# Q15. Write a final processing pipeline that filters out duplicates, fills null prices with 0, and groups by store_id to calculate total revenue.

In [32]:
from pyspark.sql.functions import sum, round

final_pipeline = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(
          round(
              sum("price"),
              2
          ).alias("Total Revenue")
      )
)

final_pipeline.show()

+--------+-------------+
|store_id|Total Revenue|
+--------+-------------+
|    S004|     76194.44|
|    S001|     51383.36|
|    S002|     81721.86|
|    S005|     76957.89|
|    S003|     80543.45|
+--------+-------------+



# Conclusion

This project demonstrated the practical implementation of Apache Spark (PySpark) for data cleaning, transformation, filtering, aggregation, and analysis using a custom retail sales dataset.

Throughout the assignment, various Spark operations such as handling missing values, removing duplicate records, filtering data, grouping, aggregation, timestamp conversion, and processing pipelines were implemented successfully.

The project highlights Spark's ability to efficiently process large datasets using distributed and in-memory computing, making it a preferred framework for modern big data analytics.

# References

- Apache Spark Documentation
- PySpark API Documentation
- Celebal Technologies Data Engineering Internship Material